In [1]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from datetime import datetime
#import warnings
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
import gc

# 设置中文字体显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 忽略警告
#warnings.filterwarnings('ignore')

## 数据处理

In [2]:
def process_new_data(df_price_new):
    """
    对新数据集应用数据处理步骤
    """
    # 创建副本避免修改原始数据
    df_processed = df_price_new.copy()
    
    print(">>> 开始处理新数据集...")
    
    # 1. 修复区域映射问题
    if '城市' in df_processed.columns and '区域' in df_processed.columns:
        df_processed['修复后区域'] = df_processed['区域'].copy()
        fix_condition = (df_processed['城市'] == 5) & (df_processed['区域'] == 126)
        df_processed.loc[fix_condition, '修复后区域'] = 132
        print("✓ 区域映射修复完成")
    
    # 2. 标准化板块
    if '板块' in df_processed.columns and '修复后区域' in df_processed.columns:
        df_processed['标准化板块'] = df_processed['板块'].copy()
        block_region_mapping = df_processed[['板块', '修复后区域']].dropna().groupby('板块')['修复后区域'].nunique()
        inconsistent_blocks = block_region_mapping[block_region_mapping > 1]
        
        if len(inconsistent_blocks) > 0:
            modified_count = 0
            for block in inconsistent_blocks.index:
                block_data = df_processed[df_processed['板块'] == block]
                region_counts = block_data['修复后区域'].value_counts()
                base_region = region_counts.index[0]
                other_regions = region_counts.index[1:]
                
                for i, region in enumerate(other_regions, 1):
                    new_block = f"{block}_{i}"
                    mask = (df_processed['板块'] == block) & (df_processed['修复后区域'] == region)
                    df_processed.loc[mask, '标准化板块'] = new_block
                    modified_count += mask.sum()
            print("✓ 板块标准化完成")

    if '板块' in df_processed.columns and '区县' in df_processed.columns:
        df_processed['标准化板块2'] = df_processed['板块'].copy()
        block_region_mapping = df_processed[['板块', '区县']].dropna().groupby('板块')['区县'].nunique()
        inconsistent_blocks = block_region_mapping[block_region_mapping > 1]
        
        if len(inconsistent_blocks) > 0:
            modified_count = 0
            for block in inconsistent_blocks.index:
                block_data = df_processed[df_processed['板块'] == block]
                region_counts = block_data['区县'].value_counts()
                base_region = region_counts.index[0]
                other_regions = region_counts.index[1:]
                
                for i, region in enumerate(other_regions, 1):
                    new_block = f"{block}_{i}"
                    mask = (df_processed['板块'] == block) & (df_processed['区县'] == region)
                    df_processed.loc[mask, '标准化板块2'] = new_block
                    modified_count += mask.sum()
            print("✓ 板块标准化完成")
    
    # 3. 提取建筑面积和套内面积数值
    if '建筑面积' in df_processed.columns:
        df_processed['建筑面积_数值'] = df_processed['建筑面积'].str.extract(r'(\d+\.?\d*)').astype(float)
        print("✓ 建筑面积数值提取完成")
    
    if '套内面积' in df_processed.columns:
        df_processed['套内面积_数值'] = df_processed['套内面积'].str.extract(r'(\d+\.?\d*)').astype(float)
        print("✓ 套内面积数值提取完成")
    
    # 4. 处理建筑年代和房龄
    def parse_build_year_middle(text_data):
        if not isinstance(text_data, str):
            return np.nan
        numbers = re.findall(r'(\d{4})', text_data)
        if len(numbers) == 0:
            return np.nan
        elif len(numbers) == 1:
            return float(numbers[0])
        else:
            first_year = float(numbers[0])
            last_year = float(numbers[-1])
            return (first_year + last_year) / 2
    
    if '建筑年代' in df_processed.columns:
        df_processed['建筑年代_中值'] = df_processed['建筑年代'].apply(parse_build_year_middle)
        df_processed['房龄_中值年份'] = 2025 - df_processed['建筑年代_中值']
        
        median_build_year = df_processed['建筑年代_中值'].median()
        median_house_age = df_processed['房龄_中值年份'].median()
        
        df_processed['建筑年代_中值'] = df_processed['建筑年代_中值'].fillna(median_build_year)
        df_processed['房龄_中值年份'] = df_processed['房龄_中值年份'].fillna(median_house_age)
        print("✓ 建筑年代处理完成")
    
    # 5. 处理物业费
    def parse_property_fee(text_data):
        if not isinstance(text_data, str):
            return np.nan
        numbers = re.findall(r'(\d+\.?\d*)', text_data)
        numbers = [num for num in numbers if num.strip()]
        if len(numbers) == 0:
            return np.nan
        elif len(numbers) == 1:
            return float(numbers[0])
        else:
            first_value = float(numbers[0])
            last_value = float(numbers[-1])
            return round((first_value + last_value) / 2, 2)
    
    if '物 业 费' in df_processed.columns:
        df_processed['物业费_中值'] = df_processed['物 业 费'].apply(parse_property_fee)
        median_fee = df_processed['物业费_中值'].median()
        df_processed['物业费_中值'] = df_processed['物业费_中值'].fillna(median_fee)
        print("✓ 物业费处理完成")
    
    # 6. 提取房屋总数数值
    if '房屋总数' in df_processed.columns:
        df_processed['房屋总数_数值'] = df_processed['房屋总数'].str.extract(r'(\d+)').astype(float)
        print("✓ 房屋总数数值提取完成")
    
    # 7. 处理交易时间相关特征
    if '交易时间' in df_processed.columns:
        df_processed['交易时间'] = pd.to_datetime(df_processed['交易时间'])
        df_processed['交易时间年份'] = df_processed['交易时间'].dt.year
        df_processed['交易月份'] = df_processed['交易时间'].dt.month
        df_processed['交易日期'] = df_processed['交易时间'].dt.day
        
        def get_period_of_month(day):
            if 1 <= day <= 10:
                return '上旬'
            elif 11 <= day <= 20:
                return '中旬'
            else:
                return '下旬'
        
        df_processed['交易日期_上中下旬'] = df_processed['交易日期'].apply(get_period_of_month)
        
        if '上次交易' in df_processed.columns:
            df_processed['上次交易'] = pd.to_datetime(df_processed['上次交易'], errors='coerce')
            
            def calculate_transaction_interval(current_time, last_time):
                if pd.notna(current_time) and pd.notna(last_time) and last_time < current_time:
                    months_diff = (current_time.year - last_time.year) * 12 + (current_time.month - last_time.month)
                    if current_time.day < last_time.day:
                        months_diff -= 1
                    return max(0, months_diff)
                else:
                    return 0
            
            df_processed['交易间隔时间'] = df_processed.apply(
                lambda row: calculate_transaction_interval(row['交易时间'], row['上次交易']), 
                axis=1
            )
            
            df_processed['是否上次交易'] = df_processed['上次交易'].apply(lambda x: 1 if pd.notna(x) else 0)
        
        print("✓ 交易时间特征处理完成")
    
    # 8. 解析房屋户型
    def parse_house_layout(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan, np.nan, np.nan, np.nan
        
        room_count = np.nan
        hall_count = np.nan
        kitchen_count = np.nan
        bathroom_count = np.nan
        
        room_match = re.search(r'(\d+)(?:室|房间)', text_data)
        if room_match:
            room_count = float(room_match.group(1))
        
        hall_match = re.search(r'(\d+)厅', text_data)
        if hall_match:
            hall_count = float(hall_match.group(1))
        
        kitchen_match = re.search(r'(\d+)厨', text_data)
        if kitchen_match:
            kitchen_count = float(kitchen_match.group(1))
        
        bathroom_match = re.search(r'(\d+)卫', text_data)
        if bathroom_match:
            bathroom_count = float(bathroom_match.group(1))
        
        return room_count, hall_count, kitchen_count, bathroom_count
    
    if '房屋户型' in df_processed.columns:
        layout_data = df_processed['房屋户型'].apply(parse_house_layout)
        df_processed['房屋户型_室_中值'] = layout_data.apply(lambda x: x[0])
        df_processed['房屋户型_厅_中值'] = layout_data.apply(lambda x: x[1])
        df_processed['房屋户型_厨_中值'] = layout_data.apply(lambda x: x[2])
        df_processed['房屋户型_卫_中值'] = layout_data.apply(lambda x: x[3])
        
        median_room = df_processed['房屋户型_室_中值'].median()
        median_hall = df_processed['房屋户型_厅_中值'].median()
        median_kitchen = df_processed['房屋户型_厨_中值'].median()
        median_bathroom = df_processed['房屋户型_卫_中值'].median()
        
        df_processed['房屋户型_室_中值'] = df_processed['房屋户型_室_中值'].fillna(median_room)
        df_processed['房屋户型_厅_中值'] = df_processed['房屋户型_厅_中值'].fillna(median_hall)
        df_processed['房屋户型_厨_中值'] = df_processed['房屋户型_厨_中值'].fillna(median_kitchen)
        df_processed['房屋户型_卫_中值'] = df_processed['房屋户型_卫_中值'].fillna(median_bathroom)
        print("✓ 房屋户型解析完成")
    
    # 9. 解析所在楼层
    def parse_floor_info(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan, np.nan
        
        floor_position = np.nan
        total_floors = np.nan
        
        position_match = re.search(r'(地下室|底层|低楼层|中楼层|高楼层|顶层)', text_data)
        if position_match:
            floor_position = position_match.group(1)
        
        floor_match = re.search(r'共(\d+)层', text_data)
        if floor_match:
            total_floors = float(floor_match.group(1))
        
        return floor_position, total_floors
    
    if '所在楼层' in df_processed.columns:
        floor_data = df_processed['所在楼层'].apply(parse_floor_info)
        df_processed['所在楼层_地底低中高顶'] = floor_data.apply(lambda x: x[0])
        df_processed['所在楼层_总楼层数'] = floor_data.apply(lambda x: x[1])
        df_processed['所在楼层_总楼层数'] = df_processed['所在楼层_总楼层数'].fillna(1)
        
        def calculate_relative_position(position):
            position_mapping = {
                '地下室': 0, '底层': 0, '低楼层': 1/6, 
                '中楼层': 1/2, '高楼层': 5/6, '顶层': 1
            }
            if pd.isna(position) or position not in position_mapping:
                return np.nan
            return position_mapping[position]
        
        df_processed['所在楼层_相对位置'] = df_processed['所在楼层_地底低中高顶'].apply(calculate_relative_position)
        
        def estimate_floor_level(position, total_floors, relative_position):
            if position == '地下室':
                return 0
            elif position == '底层':
                return 1
            if pd.notna(total_floors) and pd.notna(relative_position):
                estimated_floor = round(total_floors * relative_position)
                return max(1, min(estimated_floor, total_floors))
            return np.nan
        
        df_processed['所在楼层_估算具体楼层'] = df_processed.apply(
            lambda row: estimate_floor_level(
                row['所在楼层_地底低中高顶'], 
                row['所在楼层_总楼层数'], 
                row['所在楼层_相对位置']
            ), 
            axis=1
        )
        
        median_position = df_processed['所在楼层_地底低中高顶'].mode()[0] if not df_processed['所在楼层_地底低中高顶'].mode().empty else '中楼层'
        median_total_floors = df_processed['所在楼层_总楼层数'].median()
        median_relative_position = df_processed['所在楼层_相对位置'].median()
        median_estimated_floor = df_processed['所在楼层_估算具体楼层'].median()
        
        df_processed['所在楼层_地底低中高顶'] = df_processed['所在楼层_地底低中高顶'].fillna(median_position)
        df_processed['所在楼层_总楼层数'] = df_processed['所在楼层_总楼层数'].fillna(median_total_floors)
        df_processed['所在楼层_相对位置'] = df_processed['所在楼层_相对位置'].fillna(median_relative_position)
        df_processed['所在楼层_估算具体楼层'] = df_processed['所在楼层_估算具体楼层'].fillna(median_estimated_floor)
        print("✓ 楼层信息解析完成")
    
    # 10. 解析梯户比例
    CHINESE_NUM_MAP = {
        '零': 0, '一': 1, '二': 2, '两': 2, '三': 3, '四': 4, '五': 5,
        '六': 6, '七': 7, '八': 8, '九': 9, '十': 10, '百': 100
    }
    
    def chinese_to_arabic(chinese_num):
        if not chinese_num or not isinstance(chinese_num, str):
            return np.nan
        if chinese_num.isdigit():
            return int(chinese_num)
        if chinese_num in CHINESE_NUM_MAP and chinese_num != '百':
            return CHINESE_NUM_MAP[chinese_num]
        
        result = 0
        temp = 0
        for char in chinese_num:
            if char in CHINESE_NUM_MAP:
                value = CHINESE_NUM_MAP[char]
                if value == 100:
                    if temp == 0:
                        temp = 1
                    result += temp * value
                    temp = 0
                elif value == 10:
                    if temp == 0:
                        temp = 1
                    result += temp * value
                    temp = 0
                else:
                    temp = value
            else:
                return np.nan
        result += temp
        return result
    
    def parse_elevator_ratio(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan, np.nan
        
        elevator_count = np.nan
        household_count = np.nan
        
        chinese_pattern = r'([零一二两三四五六七八九十百]+)梯([零一二两三四五六七八九十百]+)户'
        chinese_match = re.search(chinese_pattern, text_data)
        if chinese_match:
            elevator_chinese = chinese_match.group(1)
            household_chinese = chinese_match.group(2)
            elevator_count = chinese_to_arabic(elevator_chinese)
            household_count = chinese_to_arabic(household_chinese)
        
        if pd.isna(elevator_count) or pd.isna(household_count):
            arabic_pattern = r'(\d+)梯(\d+)户'
            arabic_match = re.search(arabic_pattern, text_data)
            if arabic_match:
                elevator_count = float(arabic_match.group(1))
                household_count = float(arabic_match.group(2))
        
        return elevator_count, household_count
    
    if '梯户比例' in df_processed.columns:
        ratio_data = df_processed['梯户比例'].apply(parse_elevator_ratio)
        df_processed['梯户比例_梯数'] = ratio_data.apply(lambda x: x[0])
        df_processed['梯户比例_户数'] = ratio_data.apply(lambda x: x[1])
        
        def calculate_ratio(elevator_count, household_count):
            if pd.isna(elevator_count) or pd.isna(household_count):
                return np.nan
            if household_count == 0:
                return np.nan
            return round(elevator_count / household_count, 4)
        
        df_processed['梯户比例_梯户比'] = df_processed.apply(
            lambda row: calculate_ratio(row['梯户比例_梯数'], row['梯户比例_户数']), 
            axis=1
        )
        
        def classify_ratio(ratio):
            if pd.isna(ratio):
                return np.nan
            if ratio >= 0.5:
                return "高梯户比"
            elif ratio >= 0.2:
                return "中梯户比"
            else:
                return "低梯户比"
        
        df_processed['梯户比例_梯户比_分类'] = df_processed['梯户比例_梯户比'].apply(classify_ratio)
        
        median_elevator = df_processed['梯户比例_梯数'].median()
        median_household = df_processed['梯户比例_户数'].median()
        median_ratio = df_processed['梯户比例_梯户比'].median()
        mode_class = df_processed['梯户比例_梯户比_分类'].mode()[0] if not df_processed['梯户比例_梯户比_分类'].mode().empty else "中梯户比"
        
        df_processed['梯户比例_梯数'] = df_processed['梯户比例_梯数'].fillna(median_elevator)
        df_processed['梯户比例_户数'] = df_processed['梯户比例_户数'].fillna(median_household)
        df_processed['梯户比例_梯户比'] = df_processed['梯户比例_梯户比'].fillna(median_ratio)
        df_processed['梯户比例_梯户比_分类'] = df_processed['梯户比例_梯户比_分类'].fillna(mode_class)
        print("✓ 梯户比例解析完成")
    
    # 11. 处理绿化率
    def parse_greening_rate(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan
        match = re.search(r'(\d+\.?\d*)%', text_data)
        if match:
            percentage_value = float(match.group(1))
            decimal_value = percentage_value / 100
            if decimal_value >= 0.8:
                return 0.8
            else:
                return round(decimal_value, 4)
        return np.nan
    
    if '绿 化 率' in df_processed.columns:
        df_processed['绿化率_数值'] = df_processed['绿 化 率'].apply(parse_greening_rate)
        median_greening_rate = df_processed['绿化率_数值'].median()
        df_processed['绿化率_数值_中值'] = df_processed['绿化率_数值'].fillna(median_greening_rate)
        df_processed['绿化率_数值_30%'] = df_processed['绿化率_数值'].fillna(0.3)
        df_processed.drop('绿化率_数值', axis=1, inplace=True)
        print("✓ 绿化率处理完成")
    
    # 12. 处理燃气费
    def parse_gas_fee(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan, np.nan
        range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)', text_data)
        if range_match:
            left_value = float(range_match.group(1))
            right_value = float(range_match.group(2))
            return left_value, right_value
        single_match = re.search(r'(\d+\.?\d*)', text_data)
        if single_match:
            value = float(single_match.group(1))
            return value, value
        return np.nan, np.nan
    
    if '燃气费' in df_processed.columns:
        gas_data = df_processed['燃气费'].apply(parse_gas_fee)
        df_processed['燃气费_左端点'] = gas_data.apply(lambda x: x[0])
        df_processed['燃气费_右端点'] = gas_data.apply(lambda x: x[1])
        
        df_processed['燃气费_数值_区间均值'] = (df_processed['燃气费_左端点'] + df_processed['燃气费_右端点']) / 2
        df_processed['燃气费_数值_区间右端'] = df_processed['燃气费_右端点']
        df_processed['燃气费_数值_区间左端'] = df_processed['燃气费_左端点']
        df_processed['燃气费_数值_区间上四分位数'] = df_processed['燃气费_左端点'] + 0.75 * (df_processed['燃气费_右端点'] - df_processed['燃气费_左端点'])
        df_processed['燃气费_数值_区间下四分位数'] = df_processed['燃气费_左端点'] + 0.25 * (df_processed['燃气费_右端点'] - df_processed['燃气费_左端点'])
        
        median_mean = df_processed['燃气费_数值_区间均值'].median()
        median_right = df_processed['燃气费_数值_区间右端'].median()
        median_left = df_processed['燃气费_数值_区间左端'].median()
        median_upper_quartile = df_processed['燃气费_数值_区间上四分位数'].median()
        median_lower_quartile = df_processed['燃气费_数值_区间下四分位数'].median()
        
        df_processed['燃气费_数值_区间均值'] = df_processed['燃气费_数值_区间均值'].fillna(median_mean)
        df_processed['燃气费_数值_区间右端'] = df_processed['燃气费_数值_区间右端'].fillna(median_right)
        df_processed['燃气费_数值_区间左端'] = df_processed['燃气费_数值_区间左端'].fillna(median_left)
        df_processed['燃气费_数值_区间上四分位数'] = df_processed['燃气费_数值_区间上四分位数'].fillna(median_upper_quartile)
        df_processed['燃气费_数值_区间下四分位数'] = df_processed['燃气费_数值_区间下四分位数'].fillna(median_lower_quartile)
        print("✓ 燃气费处理完成")
    
    # 13. 处理停车费用
    hardcoded_parking_fees = {
        "一元钱一小时，单次24小时内最高12元一次": 12.0 * 30, 
        "小区没有停车费": 0.0,
        "无固定车位不收费": 0.0,
        "售价:30万/位；租价450元/位/月": 450.0,
        "每小时2元每个月70元": 70.0, 
        "每小时2元/位 , 每个月50元/位": 50.0, 
        "露天250元/月/位，3元/时/位；室内500元/月/位，3元/时/位": np.mean([250.0, 500.0]), 
        "露天16元/小时，室内700/月": 700.0, 
        "临保2.5元/小时月保400元/月": 400.0, 
        "临保2.5元/时/位，月保400元/月/位": 400.0, 
        "临保:4元/小时/位,月保:550元/位/月": 550.0, 
        "固定车位1400元/年，非固定车位100元/月": np.mean([1400.0 / 12.0, 100.0]),
        "第一小时5块，后面1小时1块，一天15封顶": 15.0 * 30, 
        "地下400，地上免费": np.mean([400.0, 0.0]), 
        "地下350元/月/位 加60管理费/月": 350.0 + 60.0, 
        "地上免费  地下400元/月": np.mean([0.0, 400.0]), 
        "地上4元/小时/位": 4.0 * 8.0 * 30.0,
        "地上150元/月/位，地下2元/时/位，地下固定车位450元/月/位": np.mean([150.0, 2.0 * 8.0 * 30.0, 450.0]),
        "地上70元/月  地下200元/月": np.mean([70.0, 200.0])
    }
    
    def parse_parking_fee_hardcoded(text_data):
        if not isinstance(text_data, str):
            return np.nan
        text_original = text_data.strip()
        text_lower = text_original.lower()

        if text_original in hardcoded_parking_fees:
            return hardcoded_parking_fees[text_original]

        zero_keywords = ['免费', '没有停车费', '不收费', '无停车费']
        if any(keyword in text_lower for keyword in zero_keywords) or text_lower in ['无', '暂无', '0']:
            return 0.0
            
        unknown_keywords = ['unknown', '未知', '无法核实', '无法获知', '待核实']
        if any(keyword in text_lower for keyword in unknown_keywords):
            return np.nan

        monthly_match = re.search(r'(\d+\.?\d*)\s*元\s*/\s*月', text_original)
        if monthly_match:
            return float(monthly_match.group(1))
        
        monthly_per_slot_match = re.search(r'(\d+\.?\d*)\s*元\s*/\s*位\s*/\s*月', text_original)
        if monthly_per_slot_match:
            return float(monthly_per_slot_match.group(1))
        
        yearly_match = re.search(r'(\d+\.?\d*)\s*元\s*/\s*年', text_original)
        if yearly_match:
            return float(yearly_match.group(1)) / 12.0

        hourly_match = re.search(r'(\d+\.?\d*)\s*元\s*/\s*小时', text_original)
        if hourly_match:
            hourly_rate = float(hourly_match.group(1))
            return hourly_rate * 8 * 30
        
        hourly_match2 = re.search(r'(\d+\.?\d*)\s*元\s*/\s*时', text_original)
        if hourly_match2:
            hourly_rate = float(hourly_match2.group(1))
            return hourly_rate * 8 * 30

        numbers = re.findall(r'(\d+\.?\d*)', text_original)
        numbers = [float(n) for n in numbers if n] 
        
        if len(numbers) > 0:
            reasonable_numbers = [n for n in numbers if n < 10000]
            if reasonable_numbers:
                return np.mean(reasonable_numbers)
        
        return np.nan
    
    if '停车费用' in df_processed.columns:
        df_processed['停车费用_数值'] = df_processed['停车费用'].apply(parse_parking_fee_hardcoded)
        median_parking_fee = df_processed['停车费用_数值'].median()
        df_processed['停车费用_数值'] = df_processed['停车费用_数值'].fillna(median_parking_fee)
        print("✓ 停车费用处理完成")
    
    # 14. 处理楼栋总数
    def parse_building_count(text_data):
        if not isinstance(text_data, str) or not text_data.strip():
            return np.nan
        text_clean = text_data.strip()
        if not text_clean:
            return np.nan
        number_match = re.search(r'(\d+)', text_clean)
        if number_match:
            return float(number_match.group(1))
        if '栋' in text_clean:
            return np.nan
        return np.nan
    
    if '楼栋总数' in df_processed.columns:
        df_processed['楼栋总数_数值'] = df_processed['楼栋总数'].apply(parse_building_count)
        median_building_count = df_processed['楼栋总数_数值'].median()
        df_processed['楼栋总数_数值_中值'] = df_processed['楼栋总数_数值'].fillna(median_building_count)
        df_processed.drop('楼栋总数_数值', axis=1, inplace=True)
        print("✓ 楼栋总数处理完成")
    
    # 15. 计算建筑面积平方项
    if '建筑面积_数值' in df_processed.columns:
        df_processed['建筑面积_数值_平方项'] = df_processed['建筑面积_数值'] ** 2
        print("✓ 建筑面积平方项计算完成")
    
    # 16. 构建房屋用途_别墅类型复合特征
    def create_property_type_villa_type(row):
        house_usage = row['房屋用途']
        villa_type = row['别墅类型']
        
        if house_usage == '别墅':
            if pd.isna(villa_type):
                return '别墅_NaN'
            else:
                return f'别墅_{villa_type}'
        else:
            return house_usage
    
    if '房屋用途' in df_processed.columns and '别墅类型' in df_processed.columns:
        df_processed['房屋用途_别墅类型'] = df_processed.apply(create_property_type_villa_type, axis=1)
        print("✓ 房屋用途别墅类型复合特征构建完成")
    
    # 17. 构建房屋年限_准确特征
    def create_accurate_house_age(row):
        house_age = row['房屋年限']
        house_advantage = row['房屋优势']
        transaction_interval = row['交易间隔时间']
        
        if pd.notna(house_age):
            return house_age
        
        if pd.notna(house_advantage) and isinstance(house_advantage, str):
            if '房本满五年' in house_advantage:
                return '满五年'
            if '房本满两年' in house_advantage:
                return '满两年'
        
        if pd.notna(transaction_interval):
            if transaction_interval >= 60:
                return '满五年'
            if transaction_interval >= 24:
                return '满两年'
        
        return np.nan
    
    if '房屋年限' in df_processed.columns and '房屋优势' in df_processed.columns and '交易间隔时间' in df_processed.columns:
        df_processed['房屋年限_准确'] = df_processed.apply(create_accurate_house_age, axis=1)
        print("✓ 房屋年限准确特征构建完成")
    
    # 18. 识别关键词特征
    def identify_keywords(row):
        result = {}
        required_columns = ['房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行']
        
        all_text = ""
        for col in required_columns:
            if pd.notna(row[col]) and isinstance(row[col], str):
                all_text += " " + row[col]
        
        if not all_text.strip():
            for key in ['地铁', '医院', '幼儿园', '公园', '广场', '大学']:
                result[key] = np.nan
            return result
        
        keyword_dict = {
            '地铁': ['地铁', '轨道交通'],
            '医院': ['医院', '医疗', '诊所', '卫生院'],
            '幼儿园': ['幼儿园', '幼稚园', '托儿所', '学前班', '早教', '儿童园'],
            '公园': ['公园', '绿地', '园林', '花园'],
            '广场': ['广场'],
            '大学': ['大学', '学院', '高校', '高等院校','校区', '高等学府']
        }
        
        for key, keywords in keyword_dict.items():
            found = False
            for keyword in keywords:
                if keyword in all_text:
                    found = True
                    break
            if found:
                result[key] = '有'
            else:
                result[key] = '无'
        
        return result
    
    required_text_columns = ['房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行']
    if all(col in df_processed.columns for col in required_text_columns):
        keyword_results = df_processed.apply(identify_keywords, axis=1)
        for key in ['地铁', '医院', '幼儿园', '公园', '广场', '大学']:
            df_processed[key] = keyword_results.apply(lambda x: x[key])
        print("✓ 关键词识别完成")
    
    print(f"\n>>> 新数据集处理完成！")
    print(f"处理前形状: {df_price_new.shape}")
    print(f"处理后形状: {df_processed.shape}")
    print(f"新增特征列数量: {df_processed.shape[1] - df_price_new.shape[1]}")
    
    return df_processed

## 房价预测

In [3]:
print("开始房价预测...")
def load_file(paths):
    for path in paths:
        if os.path.exists(path):
            return pd.read_csv(path, encoding='utf-8')
    return None

# 加载文件
df_price_train = load_file([
    'data/ruc_Class25Q2_train_price.csv',
    '/home/mw/input/hackathon255769/ruc_Class25Q2_train_price.csv'
])

df_price_test = load_file([
    'data/ruc_Class25Q2_test_price.csv',
    '/home/mw/input/hackathon255769/ruc_Class25Q2_test_price.csv'
])

print(f"房价训练集原始形状: {df_price_train.shape}")
print(f"房价测试集原始形状: {df_price_test.shape}")

# 处理房价数据
df_price_train_processed = process_new_data(df_price_train)
df_price_test_processed = process_new_data(df_price_test)

print(f"房价训练集处理后形状: {df_price_train_processed.shape}")
print(f"房价测试集处理后形状: {df_price_test_processed.shape}")

开始房价预测...


/tmp/ipykernel_56/2649836615.py:5: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding='utf-8')
/tmp/ipykernel_56/2649836615.py:5: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding='utf-8')


房价训练集原始形状: (103871, 55)
房价测试集原始形状: (34017, 55)
>>> 开始处理新数据集...
✓ 区域映射修复完成
✓ 板块标准化完成
✓ 板块标准化完成
✓ 建筑面积数值提取完成
✓ 套内面积数值提取完成
✓ 建筑年代处理完成
✓ 物业费处理完成
✓ 房屋总数数值提取完成
✓ 交易时间特征处理完成
✓ 房屋户型解析完成
✓ 楼层信息解析完成
✓ 梯户比例解析完成
✓ 绿化率处理完成
✓ 燃气费处理完成
✓ 停车费用处理完成
✓ 楼栋总数处理完成
✓ 建筑面积平方项计算完成
✓ 房屋用途别墅类型复合特征构建完成
✓ 房屋年限准确特征构建完成
✓ 关键词识别完成

>>> 新数据集处理完成！
处理前形状: (103871, 55)
处理后形状: (103871, 102)
新增特征列数量: 47
>>> 开始处理新数据集...
✓ 区域映射修复完成
✓ 板块标准化完成
✓ 板块标准化完成
✓ 建筑面积数值提取完成
✓ 套内面积数值提取完成
✓ 建筑年代处理完成
✓ 物业费处理完成
✓ 房屋总数数值提取完成
✓ 交易时间特征处理完成
✓ 房屋户型解析完成
✓ 楼层信息解析完成
✓ 梯户比例解析完成
✓ 绿化率处理完成
✓ 燃气费处理完成
✓ 停车费用处理完成
✓ 楼栋总数处理完成
✓ 建筑面积平方项计算完成
✓ 房屋用途别墅类型复合特征构建完成
✓ 房屋年限准确特征构建完成
✓ 关键词识别完成

>>> 新数据集处理完成！
处理前形状: (34017, 55)
处理后形状: (34017, 102)
新增特征列数量: 47
房价训练集处理后形状: (103871, 102)
房价测试集处理后形状: (34017, 102)


In [4]:
# 定义房价特征列
columns_to_keep_test_price = [
    'ID', '房屋户型_室_中值', '房屋户型_厅_中值', '房屋户型_厨_中值', 
    '房屋户型_卫_中值', '建筑结构', '建筑结构_comm', '建筑面积_数值', 
    '建筑面积_数值_平方项', '所在楼层_地底低中高顶', '所在楼层_总楼层数', 
    '所在楼层_估算具体楼层', '装修情况', '配备电梯', '房屋用途_别墅类型', 
    '交易时间年份', '交易月份', '交易日期_上中下旬', '交易权属', 
    '是否上次交易', '交易间隔时间', '房屋年限_准确', '产权所属', '产权描述', '物业类别', 
    '建筑年代_中值', '房屋总数_数值', '楼栋总数_数值_中值', '物业费_中值', 
    '停车费用_数值', '停车位', '绿化率_数值_中值',  '容 积 率', '供水', '供暖', '供电', 
    '燃气费_数值_区间均值', '房龄_中值年份', '梯户比例_梯数', '梯户比例_户数', 
    '梯户比例_梯户比_分类', '地铁', '医院', '幼儿园', '公园', '广场', '大学'
    ,'标准化板块'
]

columns_to_keep_train_price = columns_to_keep_test_price.copy()
columns_to_keep_train_price.remove('ID')
columns_to_keep_train_price.append('Price')

# 只保留存在的列
available_columns_test_price = [col for col in columns_to_keep_test_price if col in df_price_test_processed.columns]
available_columns_train_price = [col for col in columns_to_keep_train_price if col in df_price_train_processed.columns]

df_price_test_final = df_price_test_processed[available_columns_test_price].copy()
df_price_train_final = df_price_train_processed[available_columns_train_price].copy()

print(f"房价训练集最终形状: {df_price_train_final.shape}")
print(f"房价测试集最终形状: {df_price_test_final.shape}")

房价训练集最终形状: (103871, 48)
房价测试集最终形状: (34017, 48)


In [5]:
print("开始房价模型训练...")

# 准备建模数据
X_train_full = df_price_train_final.drop(columns=['Price'], errors='ignore')
y_train_full = df_price_train_final['Price']
X_test_final = df_price_test_final.drop(columns=['ID'], errors='ignore')

print(f"训练集特征形状: {X_train_full.shape}")
print(f"训练集目标形状: {y_train_full.shape}")
print(f"测试集特征形状: {X_test_final.shape}")

# 数据分割
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.2, 
    random_state=111
)

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_val.shape}")

开始房价模型训练...
训练集特征形状: (103871, 47)
训练集目标形状: (103871,)
测试集特征形状: (34017, 47)
训练集: (83096, 47)
验证集: (20775, 47)


In [6]:
#%% 
#将数值型变量转换为类别型变量
print("开始转换数值型变量为类别型...")

# 定义要转换的列
numerical_to_categorical = ['交易时间年份', '交易月份', '建筑年代_中值']

print("转换前的数据类型:")
for col in numerical_to_categorical:
    if col in X_train.columns:
        print(f"  {col}: {X_train[col].dtype}")

# 将训练集、验证集和测试集中的这些列转换为字符串类型
for col in numerical_to_categorical:
    if col in X_train.columns:
        # 转换为字符串，同时处理可能的NaN值
        X_train[col] = X_train[col].astype(str)
        print(f"  训练集 - {col} 已转换为类别型，唯一值数量: {X_train[col].nunique()}")
    
    if col in X_val.columns:
        X_val[col] = X_val[col].astype(str)
        print(f"  验证集 - {col} 已转换为类别型")
    
    if col in X_test_final.columns:
        X_test_final[col] = X_test_final[col].astype(str)
        print(f"  测试集 - {col} 已转换为类别型")

print("\n转换后的数据类型:")
for col in numerical_to_categorical:
    if col in X_train.columns:
        print(f"  {col}: {X_train[col].dtype}")

print("\n数值型变量转换完成")

开始转换数值型变量为类别型...
转换前的数据类型:
  交易时间年份: int64
  交易月份: int64
  建筑年代_中值: float64
  训练集 - 交易时间年份 已转换为类别型，唯一值数量: 8
  验证集 - 交易时间年份 已转换为类别型
  测试集 - 交易时间年份 已转换为类别型
  训练集 - 交易月份 已转换为类别型，唯一值数量: 12
  验证集 - 交易月份 已转换为类别型
  测试集 - 交易月份 已转换为类别型
  训练集 - 建筑年代_中值 已转换为类别型，唯一值数量: 116
  验证集 - 建筑年代_中值 已转换为类别型
  测试集 - 建筑年代_中值 已转换为类别型

转换后的数据类型:
  交易时间年份: object
  交易月份: object
  建筑年代_中值: object

数值型变量转换完成


In [7]:
#%%

# 异常值检测与处理
print("开始异常值检测与处理...")

# 使用IQR方法检测异常值
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# 使用Z-score方法检测异常值
def detect_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs((data[column] - data[column].mean()) / data[column].std())
    outliers = data[z_scores > threshold]
    return outliers

# 对数值型特征进行异常值检测
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
outliers_info = {}

print("异常值检测结果:")
for col in numerical_features:
    # IQR方法
    iqr_outliers, lower, upper = detect_outliers_iqr(X_train, col)
    # Z-score方法
    z_outliers = detect_outliers_zscore(X_train, col)
    
    outliers_info[col] = {
        'iqr_outliers_count': len(iqr_outliers),
        'z_outliers_count': len(z_outliers),
        'iqr_bounds': (lower, upper),
        'total_outliers': len(iqr_outliers) + len(z_outliers)
    }
    
    print(f"  {col}: IQR异常值 {len(iqr_outliers)}个, Z-score异常值 {len(z_outliers)}个")

# 统计总异常值情况
total_iqr_outliers = sum(info['iqr_outliers_count'] for info in outliers_info.values())
total_z_outliers = sum(info['z_outliers_count'] for info in outliers_info.values())
print(f"\n总计: IQR异常值 {total_iqr_outliers}个, Z-score异常值 {total_z_outliers}个")

# 处理异常值 - 使用缩尾处理(Winsorization)
def winsorize_data(data, column, lower_quantile=0.01, upper_quantile=0.99):
    lower_bound = data[column].quantile(lower_quantile)
    upper_bound = data[column].quantile(upper_quantile)
    data[column] = np.clip(data[column], lower_bound, upper_bound)
    return data

# 对训练集进行异常值处理
print("\n开始异常值处理...")
for col in numerical_features:
    original_shape = X_train.shape[0]
    X_train = winsorize_data(X_train, col)
    print(f"  对特征 {col} 进行缩尾处理")

# 对验证集和测试集也应用相同的处理（使用训练集的边界）
for col in numerical_features:
    if col in X_val.columns:
        train_lower = X_train[col].quantile(0.01)
        train_upper = X_train[col].quantile(0.99)
        X_val[col] = np.clip(X_val[col], train_lower, train_upper)
    
    if col in X_test_final.columns:
        train_lower = X_train[col].quantile(0.01)
        train_upper = X_train[col].quantile(0.99)
        X_test_final[col] = np.clip(X_test_final[col], train_lower, train_upper)

print("异常值检测与处理完成")
print(f"异常值处理后训练集形状: {X_train.shape}")

开始异常值检测与处理...
异常值检测结果:
  房屋户型_室_中值: IQR异常值 2682个, Z-score异常值 586个
  房屋户型_厅_中值: IQR异常值 118个, Z-score异常值 118个
  房屋户型_厨_中值: IQR异常值 1631个, Z-score异常值 1631个
  房屋户型_卫_中值: IQR异常值 1083个, Z-score异常值 1083个
  建筑面积_数值: IQR异常值 3287个, Z-score异常值 1583个
  建筑面积_数值_平方项: IQR异常值 5265个, Z-score异常值 1554个
  所在楼层_总楼层数: IQR异常值 2个, Z-score异常值 135个
  所在楼层_估算具体楼层: IQR异常值 708个, Z-score异常值 545个
  是否上次交易: IQR异常值 20394个, Z-score异常值 0个
  交易间隔时间: IQR异常值 2791个, Z-score异常值 799个
  房屋总数_数值: IQR异常值 4735个, Z-score异常值 1298个
  楼栋总数_数值_中值: IQR异常值 8530个, Z-score异常值 1344个
  物业费_中值: IQR异常值 4685个, Z-score异常值 795个
  停车费用_数值: IQR异常值 8764个, Z-score异常值 1587个
  停车位: IQR异常值 3252个, Z-score异常值 1602个
  绿化率_数值_中值: IQR异常值 12523个, Z-score异常值 1920个
  容 积 率: IQR异常值 2994个, Z-score异常值 1487个
  燃气费_数值_区间均值: IQR异常值 31003个, Z-score异常值 579个
  房龄_中值年份: IQR异常值 9668个, Z-score异常值 1548个
  梯户比例_梯数: IQR异常值 5852个, Z-score异常值 1314个
  梯户比例_户数: IQR异常值 6298个, Z-score异常值 1712个

总计: IQR异常值 136265个, Z-score异常值 23220个

开始异常值处理...
  对特征 房屋户型_室_中值 进行缩尾处理
  对特征 房屋户型_厅_中值

In [8]:
print("处理缺失值...")

# 处理缺失值
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

print(f"数值型特征 ({len(numerical_features)}): {list(numerical_features)}")
print(f"类别型特征 ({len(categorical_features)}): {list(categorical_features)}")

# 填充数值型特征
for col in numerical_features:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_test_final[col] = X_test_final[col].fillna(median_val)
        print(f"  填充数值特征: {col}")

# 填充类别型特征
for col in categorical_features:
    if X_train[col].isnull().any():
        mode_val = X_train[col].mode()[0] if not X_train[col].mode().empty else 'Unknown'
        X_train[col] = X_train[col].fillna(mode_val)
        X_val[col] = X_val[col].fillna(mode_val)
        X_test_final[col] = X_test_final[col].fillna(mode_val)
        print(f"  填充类别特征: {col}")

print("缺失值处理完成")

处理缺失值...
数值型特征 (21): ['房屋户型_室_中值', '房屋户型_厅_中值', '房屋户型_厨_中值', '房屋户型_卫_中值', '建筑面积_数值', '建筑面积_数值_平方项', '所在楼层_总楼层数', '所在楼层_估算具体楼层', '是否上次交易', '交易间隔时间', '房屋总数_数值', '楼栋总数_数值_中值', '物业费_中值', '停车费用_数值', '停车位', '绿化率_数值_中值', '容 积 率', '燃气费_数值_区间均值', '房龄_中值年份', '梯户比例_梯数', '梯户比例_户数']
类别型特征 (26): ['建筑结构', '建筑结构_comm', '所在楼层_地底低中高顶', '装修情况', '配备电梯', '房屋用途_别墅类型', '交易时间年份', '交易月份', '交易日期_上中下旬', '交易权属', '房屋年限_准确', '产权所属', '产权描述', '物业类别', '建筑年代_中值', '供水', '供暖', '供电', '梯户比例_梯户比_分类', '地铁', '医院', '幼儿园', '公园', '广场', '大学', '标准化板块']
  填充数值特征: 房屋总数_数值
  填充数值特征: 停车位
  填充数值特征: 容 积 率
  填充类别特征: 建筑结构
  填充类别特征: 建筑结构_comm
  填充类别特征: 装修情况
  填充类别特征: 配备电梯
  填充类别特征: 房屋用途_别墅类型
  填充类别特征: 房屋年限_准确
  填充类别特征: 产权描述
  填充类别特征: 物业类别
  填充类别特征: 供水
  填充类别特征: 供暖
  填充类别特征: 供电
  填充类别特征: 地铁
  填充类别特征: 医院
  填充类别特征: 幼儿园
  填充类别特征: 公园
  填充类别特征: 广场
  填充类别特征: 大学
缺失值处理完成


In [9]:
# 目标变量对数变换
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print("目标变量对数变换完成")

# 类别特征编码
categorical_cols_for_encoding = X_train.select_dtypes(include=['object']).columns.tolist()

if categorical_cols_for_encoding:
    print(f"将对 {len(categorical_cols_for_encoding)} 个类别特征进行One-Hot编码")
    
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols_for_encoding, drop_first=True)
    X_val_encoded = pd.get_dummies(X_val, columns=categorical_cols_for_encoding, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test_final, columns=categorical_cols_for_encoding, drop_first=True)
    
    # 确保所有数据集具有相同的特征
    all_features = X_train_encoded.columns
    X_val_encoded = X_val_encoded.reindex(columns=all_features, fill_value=0)
    X_test_encoded = X_test_encoded.reindex(columns=all_features, fill_value=0)
    
    print(f"编码后特征数量: {X_train_encoded.shape[1]}")
else:
    X_train_encoded = X_train.copy()
    X_val_encoded = X_val.copy()
    X_test_encoded = X_test_final.copy()
    print("没有需要编码的类别特征")

# 特征缩放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_val_scaled = scaler.transform(X_val_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# 转换回DataFrame保持列名
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns, index=X_train_encoded.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val_encoded.columns, index=X_val_encoded.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns, index=X_test_encoded.index)

print("特征标准化完成")
print(f"最终训练集形状: {X_train_scaled_df.shape}")
print(f"最终验证集形状: {X_val_scaled_df.shape}")
print(f"最终测试集形状: {X_test_scaled_df.shape}")

目标变量对数变换完成
将对 26 个类别特征进行One-Hot编码
编码后特征数量: 1657
特征标准化完成
最终训练集形状: (83096, 1657)
最终验证集形状: (20775, 1657)
最终测试集形状: (34017, 1657)


In [10]:
def evaluate_model(y_true_log, y_pred_log, model_name):
    """
    评估模型性能，增加数值稳定性处理
    """
    try:
        # 转换回原始尺度，增加数值稳定性
        y_true_orig = np.expm1(y_true_log)
        y_pred_orig = np.expm1(y_pred_log)
        
        # 检查并处理无效值
        mask = np.isfinite(y_pred_orig) & np.isfinite(y_true_orig)
        y_pred_orig_clean = y_pred_orig[mask]
        y_true_orig_clean = y_true_orig[mask]
        
        if len(y_pred_orig_clean) == 0:
            print(f"警告: {model_name} 所有预测值都无效")
            return {'mae_orig': np.inf, 'rmse_orig': np.inf}
        
        # 计算指标
        mae_orig = mean_absolute_error(y_true_orig_clean, y_pred_orig_clean)
        rmse_orig = np.sqrt(mean_squared_error(y_true_orig_clean, y_pred_orig_clean))
        
        # 对数尺度指标
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
        
        print(f"=== {model_name} 性能评估 ===")
        print(f"对数尺度 - MAE: {mae_log:.4f}, RMSE: {rmse_log:.4f}")
        print(f"原始尺度 - MAE: {mae_orig:,.2f}, RMSE: {rmse_orig:,.2f}")
        print(f"有效样本数: {len(y_pred_orig_clean)}/{len(y_pred_orig)}")
        
        return {
            'mae_log': mae_log,
            'rmse_log': rmse_log,
            'mae_orig': mae_orig,
            'rmse_orig': rmse_orig
        }
        
    except Exception as e:
        print(f"错误: {model_name} 评估失败 - {str(e)}")
        return {'mae_orig': np.inf, 'rmse_orig': np.inf}

In [11]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# 训练模型
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=0.001, max_iter=5000, random_state=111),
    'Ridge': Ridge(alpha=1.0, random_state=111),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=111)
}

performances = {}
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"训练 {name} 模型")
    print(f"{'='*50}")
    
    model.fit(X_train_scaled_df, y_train_log)
    trained_models[name] = model
    
    # 预测和评估
    y_train_pred = model.predict(X_train_scaled_df)
    y_val_pred = model.predict(X_val_scaled_df)
    
    train_performance = evaluate_model(y_train_log, y_train_pred, f"{name}训练集")
    val_performance = evaluate_model(y_val_log, y_val_pred, f"{name}验证集")
    
    performances[name] = {
        'train_mae': train_performance['mae_orig'],
        'val_mae': val_performance['mae_orig'],
        'train_rmse': train_performance['rmse_orig'],
        'val_rmse': val_performance['rmse_orig']
    }
    
    if name == 'Lasso':
        non_zero_coef = np.sum(model.coef_ != 0)
        print(f"Lasso选择的特征数量: {non_zero_coef}/{X_train_scaled_df.shape[1]}")
    elif name == 'ElasticNet':
        non_zero_coef = np.sum(model.coef_ != 0)
        print(f"ElasticNet选择的特征数量: {non_zero_coef}/{X_train_scaled_df.shape[1]}")
        print(f"ElasticNet L1比例: {model.l1_ratio}")


训练 OLS 模型
=== OLS训练集 性能评估 ===
对数尺度 - MAE: 0.1310, RMSE: 0.1803
原始尺度 - MAE: 305,101.92, RMSE: 750,982.53
有效样本数: 83096/83096
=== OLS验证集 性能评估 ===
对数尺度 - MAE: 1639694362.1592, RMSE: 175213613340.3253
原始尺度 - MAE: 309,476.77, RMSE: 769,455.21
有效样本数: 20760/20775

训练 Lasso 模型


/tmp/ipykernel_56/2634894338.py:8: RuntimeWarning: overflow encountered in expm1
  y_pred_orig = np.expm1(y_pred_log)


=== Lasso训练集 性能评估 ===
对数尺度 - MAE: 0.1407, RMSE: 0.1913
原始尺度 - MAE: 331,817.39, RMSE: 788,525.76
有效样本数: 83096/83096
=== Lasso验证集 性能评估 ===
对数尺度 - MAE: 0.1412, RMSE: 0.1926
原始尺度 - MAE: 329,361.33, RMSE: 790,593.39
有效样本数: 20775/20775
Lasso选择的特征数量: 1337/1657

训练 Ridge 模型
=== Ridge训练集 性能评估 ===
对数尺度 - MAE: 0.1307, RMSE: 0.1801
原始尺度 - MAE: 304,292.31, RMSE: 749,395.94
有效样本数: 83096/83096
=== Ridge验证集 性能评估 ===
对数尺度 - MAE: 0.1329, RMSE: 0.1846
原始尺度 - MAE: 307,780.70, RMSE: 761,697.50
有效样本数: 20775/20775

训练 ElasticNet 模型
=== ElasticNet训练集 性能评估 ===
对数尺度 - MAE: 0.1343, RMSE: 0.1842
原始尺度 - MAE: 313,951.26, RMSE: 760,792.10
有效样本数: 83096/83096
=== ElasticNet验证集 性能评估 ===
对数尺度 - MAE: 0.1350, RMSE: 0.1856
原始尺度 - MAE: 312,276.21, RMSE: 766,322.78
有效样本数: 20775/20775
ElasticNet选择的特征数量: 1458/1657
ElasticNet L1比例: 0.5


## 租金预测

In [16]:
# 模型性能比较
performance_df = pd.DataFrame(performances).T
print(f"\n房价模型性能比较表 (原始价格尺度):")
print(performance_df.round(2))

# 选择最佳模型
best_model_name = performance_df['val_mae'].idxmin()
best_model = trained_models[best_model_name]

print(f"\n最佳模型: {best_model_name}")
print(f"验证集MAE: {performance_df.loc[best_model_name, 'val_mae']:,.2f}")


房价模型性能比较表 (原始价格尺度):
            train_mae    val_mae  train_rmse   val_rmse
OLS         305101.92  309476.77   750982.53  769455.21
Lasso       331817.39  329361.33   788525.76  790593.39
Ridge       304292.31  307780.70   749395.94  761697.50
ElasticNet  313951.26  312276.21   760792.10  766322.78

最佳模型: Ridge
验证集MAE: 307,780.70


In [17]:
# 交叉验证
def cv_mae_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    y_orig = np.expm1(y)
    y_pred_orig = np.expm1(y_pred)
    return -mean_absolute_error(y_orig, y_pred_orig)

kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_scaled_df, y_train_log, 
                           cv=kf, scoring=cv_mae_scorer, n_jobs=-1)

cv_mae_scores = -cv_scores
print(f"6折交叉验证MAE: {cv_mae_scores}")
print(f"平均交叉验证MAE: {cv_mae_scores.mean():,.2f} (±{cv_mae_scores.std():,.2f})")

# 使用完整训练集重新训练最佳模型
X_full_scaled = scaler.transform(X_train_encoded)
X_full_scaled_df = pd.DataFrame(X_full_scaled, columns=X_train_encoded.columns)

final_model = best_model.__class__(**best_model.get_params())
final_model.fit(X_full_scaled_df, y_train_log)

print("最终模型训练完成")


6折交叉验证MAE: [320986.38180203 318543.12066244 316022.57434739 316867.33680957
 317468.99856219 310893.96490101]
平均交叉验证MAE: 316,797.06 (±3,068.71)
最终模型训练完成


In [18]:

# 对测试集进行预测
X_test_final_scaled = scaler.transform(X_test_encoded)
X_test_final_scaled_df = pd.DataFrame(X_test_final_scaled, columns=X_test_encoded.columns)

y_test_pred_log = final_model.predict(X_test_final_scaled_df)
y_test_pred_price = np.expm1(y_test_pred_log)

print(f"测试集预测完成，预测价格范围: {y_test_pred_price.min():,.2f} - {y_test_pred_price.max():,.2f}")

# 生成房价提交文件
price_submission = pd.DataFrame({
    'ID': df_price_test_final['ID'],
    'Price': y_test_pred_price
})
price_submission.to_csv('prediction1.csv', index=False)

print(f"\n房价预测文件已保存: prediction1.csv")
print(f"包含 {len(price_submission)} 条预测记录")
print("\n预测结果示例:")
print(price_submission.head(10))

测试集预测完成，预测价格范围: 81,697.04 - 31,029,990.45

房价预测文件已保存: prediction1.csv
包含 34017 条预测记录

预测结果示例:
        ID         Price
0  1000000  1.600581e+07
1  1000001  3.430630e+06
2  1000002  4.278494e+06
3  1000003  2.779867e+06
4  1000004  1.392856e+07
5  1000005  2.625228e+06
6  1000006  2.537415e+07
7  1000007  3.332143e+06
8  1000008  6.639944e+06
9  1000009  1.287649e+07


In [19]:
print("开始租金预测...")
def load_file(paths):
    for path in paths:
        if os.path.exists(path):
            return pd.read_csv(path, encoding='utf-8')
    return None

# 加载文件
df_rent_train = load_file([
    'data/ruc_Class25Q2_train_rent.csv',
    '/home/mw/input/hackathon255769/ruc_Class25Q2_train_rent.csv'
])

df_rent_test = load_file([
    'data/ruc_Class25Q2_test_rent.csv',
    '/home/mw/input/hackathon255769/ruc_Class25Q2_test_rent.csv'
])

print(f"租金训练集原始形状: {df_rent_train.shape}")
print(f"租金测试集原始形状: {df_rent_test.shape}")

# 处理租金数据
df_rent_train_processed = process_new_data(df_rent_train)
df_rent_test_processed = process_new_data(df_rent_test)

print(f"租金训练集处理后形状: {df_rent_train_processed.shape}")
print(f"租金测试集处理后形状: {df_rent_test_processed.shape}")

开始租金预测...


/tmp/ipykernel_56/2110048536.py:5: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, encoding='utf-8')


租金训练集原始形状: (98899, 46)
租金测试集原始形状: (9773, 46)
>>> 开始处理新数据集...
✓ 板块标准化完成
✓ 建筑年代处理完成
✓ 物业费处理完成
✓ 房屋总数数值提取完成
✓ 交易时间特征处理完成
✓ 绿化率处理完成
✓ 燃气费处理完成
✓ 停车费用处理完成
✓ 楼栋总数处理完成

>>> 新数据集处理完成！
处理前形状: (98899, 46)
处理后形状: (98899, 66)
新增特征列数量: 20
>>> 开始处理新数据集...
✓ 板块标准化完成
✓ 建筑年代处理完成
✓ 物业费处理完成
✓ 房屋总数数值提取完成
✓ 交易时间特征处理完成
✓ 绿化率处理完成
✓ 燃气费处理完成
✓ 停车费用处理完成
✓ 楼栋总数处理完成

>>> 新数据集处理完成！
处理前形状: (9773, 46)
处理后形状: (9773, 66)
新增特征列数量: 20
租金训练集处理后形状: (98899, 66)
租金测试集处理后形状: (9773, 66)


In [20]:
# 定义租金特征列

columns_to_keep_test_rent = [
    'ID', '房屋户型_室_中值', '房屋户型_厅_中值', '房屋户型_厨_中值', 
    '房屋户型_卫_中值', '建筑结构', '建筑结构_comm', '建筑面积_数值', 
    '建筑面积_数值_平方项', '所在楼层_地底低中高顶', '所在楼层_总楼层数', 
    '所在楼层_相对位置', '所在楼层_估算具体楼层', '装修情况', '配备电梯', '房屋用途_别墅类型', 
    '交易时间年份', '交易月份', '交易日期_上中下旬', '交易权属', 
    '是否上次交易', '交易间隔时间', '房屋年限_准确', '产权所属', '产权描述', '物业类别', 
    '建筑年代_中值', '房屋总数_数值', '楼栋总数_数值_中值', '物业费_中值', 
    '停车费用_数值', '停车位', '绿化率_数值_中值',  '容积率', '供水', '供暖', '供电', 
    '燃气费_数值_区间均值', '房龄_中值年份', '梯户比例_梯数', '梯户比例_户数', 
    '梯户比例_梯户比_分类', '地铁', '医院', '幼儿园', '公园', '广场', '大学'
    ,'付款方式', '租赁方式', '车位'
    ,'标准化板块2'
]

columns_to_keep_train_rent = columns_to_keep_test_rent.copy()
columns_to_keep_train_rent.remove('ID')
columns_to_keep_train_rent.append('Price')

# 只保留存在的列
available_columns_test_rent = [col for col in columns_to_keep_test_rent if col in df_rent_test_processed.columns]
available_columns_train_rent = [col for col in columns_to_keep_train_rent if col in df_rent_train_processed.columns]

df_rent_test_final = df_rent_test_processed[available_columns_test_rent].copy()
df_rent_train_final = df_rent_train_processed[available_columns_train_rent].copy()

print(f"租金训练集最终形状: {df_rent_train_final.shape}")
print(f"租金测试集最终形状: {df_rent_test_final.shape}")

租金训练集最终形状: (98899, 23)
租金测试集最终形状: (9773, 23)


In [21]:
print("开始租金预测...")

# 准备建模数据
X_train_full_rent = df_rent_train_final.drop(columns=['Price'], errors='ignore')
y_train_full_rent = df_rent_train_final['Price']
X_test_final_rent = df_rent_test_final.drop(columns=['ID'], errors='ignore')

print(f"训练集特征形状: {X_train_full_rent.shape}")
print(f"训练集目标形状: {y_train_full_rent.shape}")
print(f"测试集特征形状: {X_test_final_rent.shape}")

# 数据分割
X_train_rent, X_val_rent, y_train_rent, y_val_rent = train_test_split(
    X_train_full_rent, y_train_full_rent, 
    test_size=0.2, 
    random_state=111
)

print(f"训练集: {X_train_rent.shape}")
print(f"验证集: {X_val_rent.shape}")

# 转换数值型变量为类别型
print("开始转换数值型变量为类别型...")
numerical_to_categorical_rent = ['交易时间年份', '交易月份', '建筑年代_中值']

print("转换前的数据类型:")
for col in numerical_to_categorical_rent:
    if col in X_train_rent.columns:
        print(f"  {col}: {X_train_rent[col].dtype}")

# 将训练集、验证集和测试集中的这些列转换为字符串类型
for col in numerical_to_categorical_rent:
    if col in X_train_rent.columns:
        X_train_rent[col] = X_train_rent[col].astype(str)
        print(f"  训练集 - {col} 已转换为类别型，唯一值数量: {X_train_rent[col].nunique()}")
    
    if col in X_val_rent.columns:
        X_val_rent[col] = X_val_rent[col].astype(str)
        print(f"  验证集 - {col} 已转换为类别型")
    
    if col in X_test_final_rent.columns:
        X_test_final_rent[col] = X_test_final_rent[col].astype(str)
        print(f"  测试集 - {col} 已转换为类别型")

print("数值型变量转换完成")

# 异常值检测与处理
print("开始异常值检测与处理...")

# 对数值型特征进行异常值检测
numerical_features_rent = X_train_rent.select_dtypes(include=['int64', 'float64']).columns
outliers_info_rent = {}

print("异常值检测结果:")
for col in numerical_features_rent:
    # IQR方法
    iqr_outliers, lower, upper = detect_outliers_iqr(X_train_rent, col)
    # Z-score方法
    z_outliers = detect_outliers_zscore(X_train_rent, col)
    
    outliers_info_rent[col] = {
        'iqr_outliers_count': len(iqr_outliers),
        'z_outliers_count': len(z_outliers),
        'iqr_bounds': (lower, upper),
        'total_outliers': len(iqr_outliers) + len(z_outliers)
    }
    
    print(f"  {col}: IQR异常值 {len(iqr_outliers)}个, Z-score异常值 {len(z_outliers)}个")

# 处理异常值 - 使用缩尾处理
print("\n开始异常值处理...")
for col in numerical_features_rent:
    original_shape = X_train_rent.shape[0]
    X_train_rent = winsorize_data(X_train_rent, col)
    print(f"  对特征 {col} 进行缩尾处理")

# 对验证集和测试集也应用相同的处理
for col in numerical_features_rent:
    if col in X_val_rent.columns:
        train_lower = X_train_rent[col].quantile(0.01)
        train_upper = X_train_rent[col].quantile(0.99)
        X_val_rent[col] = np.clip(X_val_rent[col], train_lower, train_upper)
    
    if col in X_test_final_rent.columns:
        train_lower = X_train_rent[col].quantile(0.01)
        train_upper = X_train_rent[col].quantile(0.99)
        X_test_final_rent[col] = np.clip(X_test_final_rent[col], train_lower, train_upper)

print("异常值检测与处理完成")

# 处理缺失值
print("处理租金数据缺失值...")

numerical_features_rent = X_train_rent.select_dtypes(include=['int64', 'float64']).columns
categorical_features_rent = X_train_rent.select_dtypes(include=['object', 'category']).columns

print(f"数值型特征 ({len(numerical_features_rent)}): {list(numerical_features_rent)}")
print(f"类别型特征 ({len(categorical_features_rent)}): {list(categorical_features_rent)}")

# 填充数值型特征
for col in numerical_features_rent:
    if X_train_rent[col].isnull().any():
        median_val = X_train_rent[col].median()
        X_train_rent[col] = X_train_rent[col].fillna(median_val)
        X_val_rent[col] = X_val_rent[col].fillna(median_val)
        X_test_final_rent[col] = X_test_final_rent[col].fillna(median_val)
        print(f"  填充数值特征: {col}")

# 填充类别型特征
for col in categorical_features_rent:
    if X_train_rent[col].isnull().any():
        mode_val = X_train_rent[col].mode()[0] if not X_train_rent[col].mode().empty else 'Unknown'
        X_train_rent[col] = X_train_rent[col].fillna(mode_val)
        X_val_rent[col] = X_val_rent[col].fillna(mode_val)
        X_test_final_rent[col] = X_test_final_rent[col].fillna(mode_val)
        print(f"  填充类别特征: {col}")

print("租金数据缺失值处理完成")

# 目标变量对数变换
y_train_log_rent = np.log1p(y_train_rent)
y_val_log_rent = np.log1p(y_val_rent)

print("租金目标变量对数变换完成")

# 类别特征编码
categorical_cols_for_encoding_rent = X_train_rent.select_dtypes(include=['object']).columns.tolist()

if categorical_cols_for_encoding_rent:
    print(f"将对 {len(categorical_cols_for_encoding_rent)} 个类别特征进行One-Hot编码")
    
    X_train_encoded_rent = pd.get_dummies(X_train_rent, columns=categorical_cols_for_encoding_rent, drop_first=True)
    X_val_encoded_rent = pd.get_dummies(X_val_rent, columns=categorical_cols_for_encoding_rent, drop_first=True)
    X_test_encoded_rent = pd.get_dummies(X_test_final_rent, columns=categorical_cols_for_encoding_rent, drop_first=True)
    
    # 确保所有数据集具有相同的特征
    all_features_rent = X_train_encoded_rent.columns
    X_val_encoded_rent = X_val_encoded_rent.reindex(columns=all_features_rent, fill_value=0)
    X_test_encoded_rent = X_test_encoded_rent.reindex(columns=all_features_rent, fill_value=0)
    
    print(f"编码后特征数量: {X_train_encoded_rent.shape[1]}")
else:
    X_train_encoded_rent = X_train_rent.copy()
    X_val_encoded_rent = X_val_rent.copy()
    X_test_encoded_rent = X_test_final_rent.copy()
    print("没有需要编码的类别特征")

# 特征缩放
scaler_rent = StandardScaler()
X_train_scaled_rent = scaler_rent.fit_transform(X_train_encoded_rent)
X_val_scaled_rent = scaler_rent.transform(X_val_encoded_rent)
X_test_scaled_rent = scaler_rent.transform(X_test_encoded_rent)

# 转换回DataFrame保持列名
X_train_scaled_df_rent = pd.DataFrame(X_train_scaled_rent, columns=X_train_encoded_rent.columns, index=X_train_encoded_rent.index)
X_val_scaled_df_rent = pd.DataFrame(X_val_scaled_rent, columns=X_val_encoded_rent.columns, index=X_val_encoded_rent.index)
X_test_scaled_df_rent = pd.DataFrame(X_test_scaled_rent, columns=X_test_encoded_rent.columns, index=X_test_encoded_rent.index)

print("租金特征标准化完成")
print(f"最终训练集形状: {X_train_scaled_df_rent.shape}")
print(f"最终验证集形状: {X_val_scaled_df_rent.shape}")
print(f"最终测试集形状: {X_test_scaled_df_rent.shape}")

开始租金预测...
训练集特征形状: (98899, 22)
训练集目标形状: (98899,)
测试集特征形状: (9773, 22)
训练集: (79119, 22)
验证集: (19780, 22)
开始转换数值型变量为类别型...
转换前的数据类型:
  交易时间年份: int64
  交易月份: int64
  建筑年代_中值: float64
  训练集 - 交易时间年份 已转换为类别型，唯一值数量: 2
  验证集 - 交易时间年份 已转换为类别型
  测试集 - 交易时间年份 已转换为类别型
  训练集 - 交易月份 已转换为类别型，唯一值数量: 12
  验证集 - 交易月份 已转换为类别型
  测试集 - 交易月份 已转换为类别型
  训练集 - 建筑年代_中值 已转换为类别型，唯一值数量: 116
  验证集 - 建筑年代_中值 已转换为类别型
  测试集 - 建筑年代_中值 已转换为类别型
数值型变量转换完成
开始异常值检测与处理...
异常值检测结果:
  房屋总数_数值: IQR异常值 5624个, Z-score异常值 777个
  楼栋总数_数值_中值: IQR异常值 7937个, Z-score异常值 1178个
  物业费_中值: IQR异常值 5427个, Z-score异常值 747个
  停车费用_数值: IQR异常值 4854个, Z-score异常值 2051个
  停车位: IQR异常值 4713个, Z-score异常值 2167个
  绿化率_数值_中值: IQR异常值 14226个, Z-score异常值 2111个
  燃气费_数值_区间均值: IQR异常值 755个, Z-score异常值 514个
  房龄_中值年份: IQR异常值 6224个, Z-score异常值 1121个

开始异常值处理...
  对特征 房屋总数_数值 进行缩尾处理
  对特征 楼栋总数_数值_中值 进行缩尾处理
  对特征 物业费_中值 进行缩尾处理
  对特征 停车费用_数值 进行缩尾处理
  对特征 停车位 进行缩尾处理
  对特征 绿化率_数值_中值 进行缩尾处理
  对特征 燃气费_数值_区间均值 进行缩尾处理
  对特征 房龄_中值年份 进行缩尾处理
异常值检测与处理完成
处理租金数据缺失值...
数值型特征 (8)

In [22]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

In [23]:
# 训练租金模型
models_rent = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=0.001, max_iter=5000, random_state=111),
    'Ridge': Ridge(alpha=1.0, random_state=111),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=111)
}

performances_rent = {}
trained_models_rent = {}

for name, model in models_rent.items():
    print(f"\n{'='*50}")
    print(f"训练租金 {name} 模型")
    print(f"{'='*50}")
    
    model.fit(X_train_scaled_df_rent, y_train_log_rent)
    trained_models_rent[name] = model
    
    # 预测和评估
    y_train_pred_rent = model.predict(X_train_scaled_df_rent)
    y_val_pred_rent = model.predict(X_val_scaled_df_rent)
    
    train_performance_rent = evaluate_model(y_train_log_rent, y_train_pred_rent, f"租金{name}训练集")
    val_performance_rent = evaluate_model(y_val_log_rent, y_val_pred_rent, f"租金{name}验证集")
    
    performances_rent[name] = {
        'train_mae': train_performance_rent['mae_orig'],
        'val_mae': val_performance_rent['mae_orig'],
        'train_rmse': train_performance_rent['rmse_orig'],
        'val_rmse': val_performance_rent['rmse_orig']
    }
    
    if name == 'Lasso':
        non_zero_coef = np.sum(model.coef_ != 0)
        print(f"Lasso选择的特征数量: {non_zero_coef}/{X_train_scaled_df_rent.shape[1]}")
    elif name == 'ElasticNet':
        non_zero_coef = np.sum(model.coef_ != 0)
        print(f"ElasticNet选择的特征数量: {non_zero_coef}/{X_train_scaled_df_rent.shape[1]}")
        print(f"ElasticNet L1比例: {model.l1_ratio}")



训练租金 OLS 模型
=== 租金OLS训练集 性能评估 ===
对数尺度 - MAE: 0.2417, RMSE: 0.3419
原始尺度 - MAE: 145,511.36, RMSE: 343,160.99
有效样本数: 79119/79119
=== 租金OLS验证集 性能评估 ===
对数尺度 - MAE: 7751958139.1543, RMSE: 927164527191.5657
原始尺度 - MAE: 147,528.13, RMSE: 349,614.97
有效样本数: 19778/19780

训练租金 Lasso 模型


/tmp/ipykernel_56/2634894338.py:8: RuntimeWarning: overflow encountered in expm1
  y_pred_orig = np.expm1(y_pred_log)


=== 租金Lasso训练集 性能评估 ===
对数尺度 - MAE: 0.2467, RMSE: 0.3481
原始尺度 - MAE: 149,424.47, RMSE: 357,337.71
有效样本数: 79119/79119
=== 租金Lasso验证集 性能评估 ===
对数尺度 - MAE: 0.2504, RMSE: 0.3535
原始尺度 - MAE: 151,581.96, RMSE: 364,906.02
有效样本数: 19780/19780
Lasso选择的特征数量: 1269/1510

训练租金 Ridge 模型
=== 租金Ridge训练集 性能评估 ===
对数尺度 - MAE: 0.2406, RMSE: 0.3409
原始尺度 - MAE: 144,708.68, RMSE: 342,160.56
有效样本数: 79119/79119
=== 租金Ridge验证集 性能评估 ===
对数尺度 - MAE: 0.2451, RMSE: 0.3476
原始尺度 - MAE: 147,257.92, RMSE: 350,215.35
有效样本数: 19780/19780

训练租金 ElasticNet 模型
=== 租金ElasticNet训练集 性能评估 ===
对数尺度 - MAE: 0.2428, RMSE: 0.3438
原始尺度 - MAE: 146,650.36, RMSE: 350,187.92
有效样本数: 79119/79119
=== 租金ElasticNet验证集 性能评估 ===
对数尺度 - MAE: 0.2466, RMSE: 0.3493
原始尺度 - MAE: 148,770.39, RMSE: 358,050.13
有效样本数: 19780/19780
ElasticNet选择的特征数量: 1371/1510
ElasticNet L1比例: 0.5


In [24]:

# 模型性能比较
performance_df_rent = pd.DataFrame(performances_rent).T
print(f"\n租金模型性能比较表 (原始价格尺度):")
print(performance_df_rent.round(2))

# 选择最佳模型
best_model_name_rent = performance_df_rent['val_mae'].idxmin()
best_model_rent = trained_models_rent[best_model_name_rent]

print(f"\n最佳模型: {best_model_name_rent}")
print(f"验证集MAE: {performance_df_rent.loc[best_model_name_rent, 'val_mae']:,.2f}")



租金模型性能比较表 (原始价格尺度):
            train_mae    val_mae  train_rmse   val_rmse
OLS         145511.36  147528.13   343160.99  349614.97
Lasso       149424.47  151581.96   357337.71  364906.02
Ridge       144708.68  147257.92   342160.56  350215.35
ElasticNet  146650.36  148770.39   350187.92  358050.13

最佳模型: Ridge
验证集MAE: 147,257.92


In [25]:
def cv_mae_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    y_orig = np.expm1(y)
    y_pred_orig = np.expm1(y_pred)
    return -mean_absolute_error(y_orig, y_pred_orig)

In [26]:

# 交叉验证
print(f"\n开始租金模型交叉验证...")
kf_rent = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_rent = cross_val_score(best_model_rent, X_train_scaled_df_rent, y_train_log_rent, 
                                cv=kf_rent, scoring=cv_mae_scorer, n_jobs=-1)

cv_mae_scores_rent = -cv_scores_rent
print(f"6折交叉验证MAE: {cv_mae_scores_rent}")
print(f"平均交叉验证MAE: {cv_mae_scores_rent.mean():,.2f} (±{cv_mae_scores_rent.std():,.2f})")

# 使用完整训练集重新训练最佳模型
X_full_scaled_rent = scaler_rent.transform(X_train_encoded_rent)
X_full_scaled_df_rent = pd.DataFrame(X_full_scaled_rent, columns=X_train_encoded_rent.columns)

final_model_rent = best_model_rent.__class__(**best_model_rent.get_params())
final_model_rent.fit(X_full_scaled_df_rent, y_train_log_rent)

print("租金最终模型训练完成")

# 对测试集进行预测
X_test_final_scaled_rent = scaler_rent.transform(X_test_encoded_rent)
X_test_final_scaled_df_rent = pd.DataFrame(X_test_final_scaled_rent, columns=X_test_encoded_rent.columns)

y_test_pred_log_rent = final_model_rent.predict(X_test_final_scaled_df_rent)
y_test_pred_rent = np.expm1(y_test_pred_log_rent)

print(f"租金测试集预测完成，预测价格范围: {y_test_pred_rent.min():,.2f} - {y_test_pred_rent.max():,.2f}")

# 生成租金提交文件
rent_submission = pd.DataFrame({
    'ID': df_rent_test_final['ID'],
    'Price': y_test_pred_rent
})
rent_submission.to_csv('prediction2.csv', index=False)

print(f"\n租金预测文件已保存: prediction2.csv")
print(f"包含 {len(rent_submission)} 条预测记录")
print("\n预测结果示例:")
print(rent_submission.head(10))



开始租金模型交叉验证...
6折交叉验证MAE: [149984.96517269 150275.96886933 150093.3495285  146030.83490344
 150366.89017447 146367.90258451]
平均交叉验证MAE: 148,853.32 (±1,883.12)
租金最终模型训练完成
租金测试集预测完成，预测价格范围: 61,745.03 - 10,341,702.22

租金预测文件已保存: prediction2.csv
包含 9773 条预测记录

预测结果示例:
        ID         Price
0  2000000  1.485084e+05
1  2000001  3.760716e+05
2  2000002  3.712166e+05
3  2000003  1.961982e+06
4  2000004  9.660862e+05
5  2000005  3.201206e+05
6  2000006  2.368428e+05
7  2000007  8.014847e+05
8  2000008  2.518339e+05
9  2000009  1.644623e+05


In [27]:
from datetime import datetime
# 合并预测结果
print("合并房价和租金预测结果...")

merged_predictions = pd.concat([
    pd.read_csv('prediction1.csv'),
    pd.read_csv('prediction2.csv')
], ignore_index=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
merged_filename = f'merged_{timestamp}.csv'
merged_predictions.to_csv(merged_filename, index=False)

print(f"合并后的预测文件已保存: merged.csv")
print(f"总记录数: {len(merged_predictions)}")


合并房价和租金预测结果...
合并后的预测文件已保存: merged.csv
总记录数: 43790
